In [ ]:
import warnings
from pathlib import Path

from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter('ignore', InterpolationWarning)

from input import input
import config
from model import generics, hybrid_system_exp, grid_search_exp
from model.feature_selection import TimeSeriesFeatureSelector
from metrics import metrics
import numpy as np

from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
import pandas as pd

%load_ext autoreload
%autoreload 2

In [ ]:
# === CELULA DE CONFIGURACAO -- Tarefa 7 (mesmo padrao ja usado pelos
# notebooks de FS de ARIMA-MLP desde a Tarefa 3.2, e de SVR single desde a
# Tarefa 6) ===
# Edite aqui para uma nova rodada: series, experiment_id, grid. Ver
# RUNBOOK.md Secao 1 (series/lag_size) e Secao 3 (convencao de experiment_id).
#
# Tarefa 7 do PLANO_ARQUITETURA.md: ultima familia da matriz completa
# (ARIMA, ARIMA-MLP, MLP, SVR, ARIMA-SVR) x 5 metodos de FS -- combina o que
# ja foi validado separadamente: Pipeline([selector, estimador]) dentro de
# Additive (Tarefas 2/3, com MLPRegressor) e SVR como estimador (Tarefa 6,
# dentro de SKlearnModel). Confirmado por pre-check (Tarefa 7) que a
# combinacao especifica Additive+SVR funciona identica, sem NENHUMA mudanca
# em hybrid_system_exp.py/single_ml_model_exp.py/grid_search_exp.py.
#
# ACHADO PROVISORIO (nao resolvido, Tarefa 6.2 -- mesmo efeito nesta
# familia): gamma='auto' do SVR e calculado como 1/n_features, entao reduzir
# features via FS muda o gamma automaticamente junto com a selecao --
# confunde "efeito de selecao de features" com "efeito de largura de
# kernel". Decisao do pesquisador: MANTER gamma='auto' por ora (nao fixar),
# a revisar com o orientador. Ver PLANO_ARQUITETURA.md Secao 1.11 (nota
# marcada como PROVISORIA, nao definitiva) -- se a decisao mudar, esta
# familia E a familia SVR single precisarao ser re-rodadas.
#
# strategy fixa nesta execucao ('rfecv') -- numero de features decidido pela CV interna do RFECV, SEM selector__k (Tarefa 4).
model = Pipeline([
    ('selector', TimeSeriesFeatureSelector(strategy='rfecv')),
    ('estimator', SVR(max_iter=100000)),
])

# Series a rodar nesta execucao (FS_DEV_SERIES por padrao -- tests/model/conftest.py).
# lag_size='auto' medido na Tarefa 7 (pre-check, contexto hibrido -- residuo
# do ARIMA, diff_kpss=False): resolve para os MESMOS valores ja medidos para
# o hibrido ARIMA-MLP (airlines=20, austres=1, coloradoRiver=16, sunspot=9)
# -- o residuo em si nao depende do estimador a jusante (MLP ou SVR),
# confirmado com dado real (ver tests/model/test_hybrid_system_exp.py).
fs_series_list = ['airlines.txt', 'austres.txt', 'coloradoRiver.txt', 'sunspot.txt']

# experiment_id novo e explicito (Secao 3.2 do CLAUDE.md) -- nunca reaproveitar.
# Mesma convencao de prefixo de familia das Tarefas 5/6: chamados_v4_fs_arimasvr_<metodo>.
experiment_id = 'chamados_v4_fs_arimasvr_rfecv'
# Caminhos derivados de experiment_id, computados uma vez aqui e reusados nas
# celulas 3/5/6 (mesmo padrao ja usado pelos notebooks de ARIMA-MLP).
experiment_dir = Path(config.MODEL_DATA_PATH) / experiment_id
experiment_dir_results = Path(config.ROOT_PATH) / 'results' / experiment_id

# Sufixo sem underscore -- extract_series_name_from_path/model_name em
# metrics.py precisam disso. Baseline hibrido SVR usa model_name='as' (sem
# prefixo de horizon, adicionado automaticamente por
# GridSearch/generics.format_names) -- aqui 'asrfecv' vira
# '1asrfecv' no nome do .pkl, mesma logica de 'amv1rfembedded' ->
# '1amv1rfembedded' no ARIMA-MLP.
model_name = 'asrfecv'
normalize = True
force = False
# SVR e deterministico (CLAUDE.md Secao 3.4) -- model_exec=1, igual ao
# baseline arima_svr.ipynb. NAO usar 10 (isso e especifico de MLP/ARIMA-MLP,
# estocasticos por inicializacao de pesos).
model_exec = 1

experiment_params = {
    'linear_model_name': '1arima',
    'diff_kpss': False,
    'horizon': 1,
}

# Convencao selector__*/estimator__* (nativa do sklearn): GridSearch usa
# clone(model).set_params(**params), que resolve essas chaves automaticamente
# via Pipeline.get_params(deep=True) -- nenhuma mudanca em grid_search_exp.py.
# Grid extraido de arima_svr.ipynb (C/gamma/kernel/epsilon/tol), para manter
# paridade de grid com o baseline.
model_parameters = {
    'estimator__C': [10, 100, 1000],
    'estimator__gamma': ['auto'],
    'estimator__kernel': ['rbf'],
    'estimator__epsilon': [0.1, 0.01, 0.001],
    'estimator__tol': [0.001],
   }

In [ ]:
# Sanity-check exigido pela Tarefa 2/5/6/7: confirma que Pipeline.get_params(deep=True)
# expoe as chaves selector__*/estimator__* que GridSearch (ParameterGrid +
# clone().set_params()) vai usar, antes de rodar o grid completo.
params = model.get_params(deep=True)
required_keys = {'selector__strategy', 'estimator__C', 'estimator__kernel'}
missing = required_keys - params.keys()
assert not missing, f"get_params(deep=True) nao expos chaves esperadas: {missing}"
print("OK -- Pipeline.get_params(deep=True) expoe:", sorted(required_keys))

# Checagem de identidade (mesmo padrao ja usado pelos demais notebooks de FS,
# achado de code-review da Tarefa 3.2): trava que a 'strategy' do seletor
# (celula 1) e o experiment_id/model_name declarados na MESMA celula
# continuam consistentes entre si.
strategy_slug = model.named_steps['selector'].strategy.replace('_', '')
assert strategy_slug in experiment_id, (
    f"strategy={model.named_steps['selector'].strategy!r} nao aparece em "
    f"experiment_id={experiment_id!r} -- confirme que voce nao mudou 'strategy' "
    "na celula de configuracao sem atualizar experiment_id/model_name."
)
assert strategy_slug in model_name, (
    f"strategy={model.named_steps['selector'].strategy!r} nao aparece em "
    f"model_name={model_name!r} -- confirme consistencia."
)
print(f"OK -- strategy={model.named_steps['selector'].strategy!r} consistente com experiment_id={experiment_id!r} e model_name={model_name!r}")

In [ ]:
# Copia o ARIMA pre-treinado (mesmo modelo, nao retreina) para o novo
# experiment_id -- Additive precisa dele sob o MESMO experiment_id da variante
# FS (ver RUNBOOK.md Secao 3). Usa copy_pretrained_linear_model (src/utils/),
# que por baixo usa generics.format_names() -- o mesmo helper que Additive/
# input_linear_info usam para localizar esse .pkl -- e le o nome do modelo
# linear de experiment_params['linear_model_name'] (celula 1), em vez de um
# literal cravado.
from utils.copy_pretrained_linear_model import copy_pretrained_linear_model

copy_pretrained_linear_model(
    source_experiment_id='chamados',
    dest_experiment_id=experiment_id,
    series_list=fs_series_list,
    linear_model_name=experiment_params['linear_model_name'],
)

In [ ]:
# Chamado direto via GridSearch(...).execution() por serie, em vez de
# grid_seach_multiple_bases(), para nao depender/mutar config.BASE_NAME_LIST
# (mesmo motivo ja documentado nos demais notebooks de FS). use_val_slipt_for_prev=True
# e explicito porque o default de GridSearch (False) diverge do default do
# wrapper grid_seach_multiple_bases (True) -- sem isso, o refit final
# perderia o val_size e os .pkl ficariam sem val_metrics, quebrando a
# paridade com o baseline original (as em chamados/).
for base_name in fs_series_list:
    print(base_name)
    exec_gs = grid_search_exp.GridSearch(
        hybrid_system_exp.Additive,
        model,
        model_parameters,
        experiment_id,
        base_name,
        model_name,
        force,
        normalize,
        experiment_params,
        model_exec=model_exec,
        use_val_slipt_for_prev=True,
    )
    exec_gs.execution()

In [ ]:
# Gera o CSV de metricas (agregado + detalhado) direto no notebook -- mesma
# funcao usada pela CLI (src/utils/export_metrics_to_csv.py), so o ponto de
# entrada muda (mesmo padrao notebook-only ja usado pelas outras familias).
from utils.export_metrics_to_csv import run_export_metrics_to_csv

metrics_output = experiment_dir_results / 'metrics.csv'
df_metrics = run_export_metrics_to_csv(
    experiment_dir,
    metrics_output,
    detail=True,
)
df_metrics

In [ ]:
# Gera o CSV de features selecionadas (agregado + detalhado) direto no
# notebook -- mesma funcao usada pela CLI (src/utils/export_selected_features.py),
# so o ponto de entrada muda (mesmo padrao notebook-only ja usado pelas outras familias).
from utils.export_selected_features import run_export_selected_features

features_output = experiment_dir_results / 'selected_features.csv'
df_features = run_export_selected_features(
    experiment_dir,
    features_output,
    detail=True,
)
df_features